# Fase 5 · M06: Ensambles

**TFM: Pronóstico del Éxito y del Abandono en los Títulos de Grado de la UJI**

| | |
|---|---|
| **Autora** | María José Morte Ruiz |
| **Institución** | UOC + Universitat Jaume I |
| **Email** | mjmorteruiz@uoc.edu · morte@uji.es |
| **Fase** | 5 — Modelado Clásico |
| **Módulo** | M06 — Ensambles |

---

## 🎯 Qué hace

Entrena y evalúa la familia de **ensambles heterogéneos** sobre D_strict
(24 features + target `abandono`, incl. indicadores de imputación informativa). Los ensambles combinan múltiples modelos
para superar el rendimiento individual:

- **VotingClassifier** — votación suave (soft voting) de los mejores modelos
  de m01–m05. Sin hiperparámetros propios: el rendimiento depende de los modelos base.
- **StackingClassifier** — meta-modelo LogReg que aprende a combinar las
  predicciones de los modelos base. Suele ser el más potente.
- **BaggingClassifier** — bootstrap aggregating con base DecisionTree.
  Reduce varianza entrenando en subconjuntos aleatorios del dataset.
- **AdaBoost** — boosting adaptativo clásico. Cada árbol corrige los errores
  del anterior dando más peso a los ejemplos mal clasificados.

> **Nota:** Random Forest y Extra Trees (bagging con árboles) están en m02.
> XGBoost, LightGBM y CatBoost (boosting moderno) están en m03.
> Este módulo completa el espectro con ensambles heterogéneos y AdaBoost clásico.

## 📋 Requisitos

- `data/05_modelado/X_train_prep.parquet`, `X_test_prep.parquet`
- Modelos base de m01–m05 en `data/05_modelado/models/` (para Voting y Stacking)
- Entorno: `tfm_abandono` (scikit-learn ≥1.3, optuna, imbalanced-learn, codecarbon)

## 📤 Genera

| Archivo | Contenido |
|---|---|
| `data/05_modelado/results/results_ensambles.parquet` | Métricas 12 combinaciones |
| `data/05_modelado/results/results_ensambles.xlsx` | Mismas métricas en Excel |
| `data/05_modelado/results/results_ensambles.json` | Metadatos + hiperparámetros |
| `data/05_modelado/models/Voting__*.pkl` | 3 modelos Voting |
| `data/05_modelado/models/Stacking__*.pkl` | 3 modelos Stacking |
| `data/05_modelado/models/Bagging__*.pkl` | 3 modelos Bagging |
| `data/05_modelado/models/AdaBoost__*.pkl` | 3 modelos AdaBoost |
| `docs/html/fase5/m06_ensambles.html` | Informe HTML |

## 🔄 Flujo

```
X_train_prep / y_train  (f5_m01_preparacion)
    ↓ Modelos base cargados de m01–m05 (Voting + Stacking)
    ↓ Optuna: Bagging 20 trials · AdaBoost 20 trials
    ↓ 5-Fold CV × 12 combinaciones (4 modelos × 3 estrategias)
    ↓ Análisis contribución modelos base (Voting/Stacking)
    ↓ Calibración + ROC + PR curve
    ↓ Ajuste umbral óptimo
    ↓ codecarbon
    → results_ensambles.parquet + .json + .pkl + HTML
```

## ➡️ Siguiente

`f5_m07_comparacion.ipynb` — comparativa global de todos los módulos


In [1]:
# ============================================================================
# CELDA 1: CONFIGURACIÓN Y DEPENDENCIAS
# ============================================================================

import sys
import warnings
import json
import time
import tracemalloc
from pathlib import Path
from datetime import datetime

import joblib
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

from sklearn.ensemble import (
    VotingClassifier, StackingClassifier,
    BaggingClassifier, AdaBoostClassifier,
    RandomForestClassifier, GradientBoostingClassifier
)
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.pipeline import Pipeline
from sklearn.model_selection import (
    StratifiedKFold, StratifiedShuffleSplit,
    cross_validate, learning_curve
)
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.metrics import (
    f1_score, roc_auc_score, precision_score, recall_score,
    accuracy_score, cohen_kappa_score, log_loss,
    confusion_matrix, classification_report,
    roc_curve, precision_recall_curve, average_precision_score
)
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

try:
    from codecarbon import EmissionsTracker
    CODECARBON_OK = True
except ImportError:
    CODECARBON_OK = False
    print('⚠️  codecarbon no instalado')

warnings.filterwarnings('ignore')

# ROOT detection
ROOT = Path.cwd()
for _ in range(6):
    if (ROOT / 'src').exists():
        break
    ROOT = ROOT.parent

sys.path.insert(0, str(ROOT))

from src.config import RUTA_HTML, info_entorno
from src.utils import crear_directorios, formato_numero_es
from src.utils.graficos import figura_a_base64
from src.html.render import render_pagina_desde_fichero

# Rutas
RUTA_MODELADO = ROOT / 'data' / '05_modelado'
RUTA_RESULTS  = RUTA_MODELADO / 'results'
RUTA_MODELS   = RUTA_MODELADO / 'models'
RUTA_HTML_F5  = RUTA_HTML / 'fase5'
crear_directorios([RUTA_RESULTS, RUTA_MODELS, RUTA_HTML_F5])

# Constantes
RANDOM_STATE      = 42
N_SPLITS_CV       = 5
N_TRIALS_OPTUNA   = 20   # solo para Bagging y AdaBoost
FAMILIA           = 'ensambles'
ESTRATEGIAS       = ['none', 'balanced', 'smote']
MODELOS_M06       = ['Voting', 'Stacking', 'Bagging', 'AdaBoost']
COLOR             = '#3182ce'
fmt               = formato_numero_es

info_entorno()
print(f'\n📂 ROOT    : {ROOT}')
print(f'📂 RESULTS : {RUTA_RESULTS}')
print(f'📂 MODELS  : {RUTA_MODELS}')
print(f'\n🌿 codecarbon: {CODECARBON_OK}')


✓ Directorios verificados: 3
✓ ===========================================================================
✓ 📌 INFORMACIÓN DEL ENTORNO DEL PROYECTO
✓ ===========================================================================
✓ 🖥️  Entorno detectado: Local
✓ 📂 Ruta base:     C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI
✓ 📁 RAW:           C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\00_raw
✓ 📁 INTERIM:       C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\01_interim
✓ 📁 PROCESSED:     C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\02_processed
✓ 📁 FEATURES:      C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\03_features
✓ 📁 AUTOML:        C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\automl
✓ 📁 NOTEBOOKS:     C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\notebooks
✓ 📄 Excel principal: C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\00_raw\datos_proyecto_sin_preinscrip.xlsx
✓

In [2]:
# ============================================================================
# CELDA 2: CARGA DE DATOS + MODELOS BASE (m01–m05)
# ============================================================================
# Voting y Stacking usan los mejores modelos entrenados en módulos anteriores.
# Si algún pkl no existe se usa un modelo por defecto equivalente.
# ============================================================================

X_train = pd.read_parquet(RUTA_MODELADO / 'X_train.parquet')
X_test  = pd.read_parquet(RUTA_MODELADO / 'X_test.parquet')
y_train = pd.read_parquet(RUTA_MODELADO / 'y_train.parquet').squeeze()
y_test  = pd.read_parquet(RUTA_MODELADO / 'y_test.parquet').squeeze()

pipeline_prep = joblib.load(RUTA_MODELADO / 'pipeline_preprocesamiento.pkl')

ruta_train_prep = RUTA_MODELADO / 'X_train_prep.parquet'
ruta_test_prep  = RUTA_MODELADO / 'X_test_prep.parquet'

if ruta_train_prep.exists() and ruta_test_prep.exists():
    X_train_prep = pd.read_parquet(ruta_train_prep)
    X_test_prep  = pd.read_parquet(ruta_test_prep)
    print('✅ Preprocesados cargados desde disco')
else:
    X_train_prep = pd.DataFrame(
        pipeline_prep.transform(X_train),
        columns=X_train.columns, index=X_train.index
    )
    X_test_prep = pd.DataFrame(
        pipeline_prep.transform(X_test),
        columns=X_test.columns, index=X_test.index
    )
    X_train_prep.to_parquet(ruta_train_prep)
    X_test_prep.to_parquet(ruta_test_prep)
    print('✅ Preprocesados generados y guardados')

FEATURE_NAMES = list(X_train.columns)
N_FEATURES    = len(FEATURE_NAMES)

# ── Cargar mejores modelos base de m01–m05 para Voting/Stacking ──
# Candidatos: mejor pkl de cada familia según resultados previos
CANDIDATOS_BASE = [
    ('LogReg',   'LogisticRegression__smote.pkl',    LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)),
    ('RF',       'RandomForest__smote.pkl',           RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE)),
    ('CatBoost', 'CatBoost__smote.pkl',               GradientBoostingClassifier(random_state=RANDOM_STATE)),
    ('EBM',      'EBM__none.pkl',                     LogisticRegression(max_iter=500, random_state=RANDOM_STATE)),
]

modelos_base = []
print('\nCargando modelos base para Voting/Stacking:')
for nombre, pkl_name, fallback in CANDIDATOS_BASE:
    ruta_pkl = RUTA_MODELS / pkl_name
    if ruta_pkl.exists():
        modelo = joblib.load(ruta_pkl)
        print(f'  ✅ {nombre}: cargado desde {pkl_name}')
    else:
        modelo = Pipeline([('model', fallback)])
        print(f'  ⚠️  {nombre}: pkl no encontrado — usando fallback {fallback.__class__.__name__}')
    modelos_base.append((nombre, modelo))

print(f'\n  {len(modelos_base)} modelos base listos para Voting/Stacking')

print('\n' + '=' * 55)
print('DATOS CARGADOS')
print('=' * 55)
print(f'X_train : {fmt(len(X_train))} × {N_FEATURES}  |  abandono: {y_train.mean()*100:.1f}%')
print(f'X_test  : {fmt(len(X_test))}  × {N_FEATURES}  |  abandono: {y_test.mean()*100:.1f}%')
print()
print('⚠️  X_test / y_test INTOCABLES — evaluación final solo en test')


✅ Preprocesados cargados desde disco

Cargando modelos base para Voting/Stacking:
  ⚠️  LogReg: pkl no encontrado — usando fallback LogisticRegression
  ✅ RF: cargado desde RandomForest__smote.pkl
  ✅ CatBoost: cargado desde CatBoost__smote.pkl
  ✅ EBM: cargado desde EBM__none.pkl

  4 modelos base listos para Voting/Stacking

DATOS CARGADOS
X_train : 26.896 × 27  |  abandono: 29.2%
X_test  : 6.725  × 27  |  abandono: 29.2%

⚠️  X_test / y_test INTOCABLES — evaluación final solo en test


In [3]:
# ============================================================================
# CELDA 2b: DIAGNÓSTICO DE PKL — estado antes de entrenar
# ============================================================================
# Muestra qué pkl existen, si son compatibles con las features actuales,
# y el tiempo estimado de entrenamiento. Sin tocar nada.
# ============================================================================

import joblib as _jl

_MODELOS_DIAG = ['Voting', 'Stacking', 'Bagging', 'AdaBoost']
_PARQUET_DIAG = RUTA_RESULTS / 'results_ensambles.parquet'

print('=' * 60)
print(f'DIAGNÓSTICO PKL — {__name__ if "__name__" else "este notebook"}')
print('=' * 60)

_ruta_meta = RUTA_MODELADO / 'meta_preparacion.json'
_n_actual  = None
if _ruta_meta.exists():
    with open(_ruta_meta) as _f:
        _n_actual = json.load(_f).get('n_features_final')
print(f'Features actuales: {_n_actual}')
print()

_hay_que_entrenar = []
for _m in _MODELOS_DIAG:
    for _e in ['none', 'balanced', 'smote']:
        _clave = f'{_m}__{_e}'
        _ruta  = RUTA_MODELS / f'{_clave}.pkl'
        if _ruta.exists():
            try:
                _modelo = _jl.load(_ruta)
                _ultimo = _modelo.steps[-1][1] if hasattr(_modelo,'steps') else _modelo
                _n_pkl  = getattr(_ultimo, 'n_features_in_', None)
                if _n_pkl is None and hasattr(_ultimo, 'estimator'):
                    _n_pkl = getattr(_ultimo.estimator, 'n_features_in_', None)
                if _n_pkl == _n_actual:
                    print(f'  ✅ {_clave}: {_n_pkl} features — OK')
                elif _n_pkl is not None:
                    print(f'  ❌ {_clave}: {_n_pkl} features — OBSOLETO')
                    _hay_que_entrenar.append(_clave)
                else:
                    print(f'  ⚠️  {_clave}: no verificable — se asume OK')
            except Exception as _ex:
                print(f'  ⚠️  {_clave}: no cargable ({str(_ex)[:40]})')
                _hay_que_entrenar.append(_clave)
        else:
            print(f'  ⏳ {_clave}: no existe — hay que entrenar')
            _hay_que_entrenar.append(_clave)

print(f'\nParquet: {"✅ existe" if _PARQUET_DIAG.exists() else "❌ no existe"}')
print()
if not _hay_que_entrenar and _PARQUET_DIAG.exists():
    print('✅ TODO EN ORDEN — cargará desde disco (rápido)')
elif not _hay_que_entrenar and not _PARQUET_DIAG.exists():
    print('ℹ️  PKL OK pero falta parquet — recalculando métricas (~5 min)')
else:
    _n = len(_hay_que_entrenar)
    print(f'⏳ HAY QUE ENTRENAR {_n} combinaciones:')
    for _c in _hay_que_entrenar: print(f'   · {_c}')
    print(f'   Tiempo estimado: ~{_n*3}-{_n*8} min (depende de CPU)')
print('=' * 60)


DIAGNÓSTICO PKL — __main__
Features actuales: 27

  ✅ Voting__none: 27 features — OK
  ✅ Voting__balanced: 27 features — OK
  ⏳ Voting__smote: no existe — hay que entrenar
  ⏳ Stacking__none: no existe — hay que entrenar
  ⏳ Stacking__balanced: no existe — hay que entrenar
  ⏳ Stacking__smote: no existe — hay que entrenar
  ⏳ Bagging__none: no existe — hay que entrenar
  ⏳ Bagging__balanced: no existe — hay que entrenar
  ⏳ Bagging__smote: no existe — hay que entrenar
  ⏳ AdaBoost__none: no existe — hay que entrenar
  ⏳ AdaBoost__balanced: no existe — hay que entrenar
  ⏳ AdaBoost__smote: no existe — hay que entrenar

Parquet: ❌ no existe

⏳ HAY QUE ENTRENAR 10 combinaciones:
   · Voting__smote
   · Stacking__none
   · Stacking__balanced
   · Stacking__smote
   · Bagging__none
   · Bagging__balanced
   · Bagging__smote
   · AdaBoost__none
   · AdaBoost__balanced
   · AdaBoost__smote
   Tiempo estimado: ~30-80 min (depende de CPU)


In [4]:
# ============================================================================
# CELDA 3: FUNCIONES AUXILIARES
# ============================================================================

CV = StratifiedKFold(n_splits=N_SPLITS_CV, shuffle=True, random_state=RANDOM_STATE)


def construir_pipeline(modelo, estrategia: str):
    """Ensambla modelo con estrategia de balance.
    Nota: VotingClassifier y StackingClassifier no soportan class_weight
    directamente — la estrategia 'balanced' usa SMOTE para ambos.
    BaggingClassifier tampoco soporta class_weight — ídem.
    AdaBoostClassifier no soporta class_weight propio — ídem.
    """
    if estrategia == 'smote':
        return ImbPipeline([
            ('smote', SMOTE(random_state=RANDOM_STATE)),
            ('model', modelo)
        ])
    return Pipeline([('model', modelo)])


def evaluar_cv(nombre: str, modelo, X, y, estrategia: str) -> dict:
    """5-Fold CV estratificado con métricas completas + tiempo + memoria."""
    pipe = construir_pipeline(modelo, estrategia)
    scoring = {'f1': 'f1', 'roc_auc': 'roc_auc',
               'precision': 'precision', 'recall': 'recall', 'accuracy': 'accuracy'}
    tracemalloc.start()
    t0     = time.time()
    cv_res = cross_validate(pipe, X, y, cv=CV, scoring=scoring,
                            return_train_score=False, n_jobs=-1)
    t_total = time.time() - t0
    _, mem_pico = tracemalloc.get_traced_memory()
    tracemalloc.stop()
    return {
        'modelo'         : nombre,
        'estrategia'     : estrategia,
        'familia'        : FAMILIA,
        'f1_mean'        : cv_res['test_f1'].mean(),
        'f1_std'         : cv_res['test_f1'].std(),
        'auc_mean'       : cv_res['test_roc_auc'].mean(),
        'auc_std'        : cv_res['test_roc_auc'].std(),
        'precision_mean' : cv_res['test_precision'].mean(),
        'recall_mean'    : cv_res['test_recall'].mean(),
        'accuracy_mean'  : cv_res['test_accuracy'].mean(),
        'tiempo_s'       : round(t_total, 3),
        'memoria_mb'     : round(mem_pico / 1024**2, 2),
    }


def calcular_metricas_test(nombre: str, pipe_fit, X_te, y_te, estrategia: str) -> dict:
    """Métricas completas sobre test para modelo ya entrenado."""
    y_pred  = pipe_fit.predict(X_te)
    y_proba = pipe_fit.predict_proba(X_te)[:, 1]
    return {
        'modelo'         : nombre,
        'estrategia'     : estrategia,
        'f1_test'        : round(f1_score(y_te, y_pred), 4),
        'auc_test'       : round(roc_auc_score(y_te, y_proba), 4),
        'auc_pr_test'    : round(average_precision_score(y_te, y_proba), 4),
        'precision_test' : round(precision_score(y_te, y_pred), 4),
        'recall_test'    : round(recall_score(y_te, y_pred), 4),
        'accuracy_test'  : round(accuracy_score(y_te, y_pred), 4),
        'kappa_test'     : round(cohen_kappa_score(y_te, y_pred), 4),
        'logloss_test'   : round(log_loss(y_te, y_proba), 4),
    }


print('✅ Funciones auxiliares definidas')


✅ Funciones auxiliares definidas


In [5]:
# ============================================================================
# CELDA 4: HIPERPARÁMETROS — Optuna solo para Bagging y AdaBoost (20 trials)
# ============================================================================
# Voting    : sin Optuna — el rendimiento depende de los modelos base, no de
#             hiperparámetros propios. Se usa soft voting (promedio de probas).
# Stacking  : sin Optuna — meta-modelo LogReg por defecto. La ganancia viene
#             de la diversidad de los modelos base, no del meta-modelo.
# Bagging   : ✅ Optuna — n_estimators (10–200) y max_samples (0.5–1.0)
# AdaBoost  : ✅ Optuna — n_estimators (50–500) y learning_rate (0.01–2.0)
#
# Cambiar FORZAR_OPTUNA = True solo la primera vez.
# ============================================================================

FORZAR_OPTUNA = False

PARAMS_OPTUNA = {
    'Voting'   : {},   # sin hiperparámetros propios
    'Stacking' : {},   # sin hiperparámetros propios
    'Bagging'  : {
        'n_estimators' : 100,
        'max_samples'  : 0.8,
        'max_features' : 1.0,
    },
    'AdaBoost' : {
        'n_estimators'  : 200,
        'learning_rate' : 0.1,
    },
}
AUC_OPTUNA = {m: None for m in MODELOS_M06}

if not FORZAR_OPTUNA:
    mejores_params = PARAMS_OPTUNA
    print('⚠️  Params iniciales — ejecutar Optuna la primera vez (FORZAR_OPTUNA = True)')
    print('   Bagging y AdaBoost serán optimizados. Voting y Stacking usan defaults.')
    for nombre, params in mejores_params.items():
        print(f'   {nombre:<12}: {params if params else "defaults (sin Optuna)"}')
else:
    import optuna
    optuna.logging.set_verbosity(optuna.logging.WARNING)

    def objetivo_bagging(trial):
        params = {
            'n_estimators' : trial.suggest_int('n_estimators', 10, 200),
            'max_samples'  : trial.suggest_float('max_samples', 0.5, 1.0),
            'max_features' : trial.suggest_float('max_features', 0.5, 1.0),
        }
        modelo = BaggingClassifier(
            estimator=DecisionTreeClassifier(max_depth=5, random_state=RANDOM_STATE),
            **params, random_state=RANDOM_STATE, n_jobs=-1
        )
        pipe = construir_pipeline(modelo, 'smote')
        return cross_validate(pipe, X_train_prep, y_train, cv=CV,
                              scoring='roc_auc', n_jobs=1)['test_score'].mean()

    def objetivo_adaboost(trial):
        params = {
            'n_estimators'  : trial.suggest_int('n_estimators', 50, 500),
            'learning_rate' : trial.suggest_float('learning_rate', 0.01, 2.0, log=True),
        }
        modelo = AdaBoostClassifier(
            estimator=DecisionTreeClassifier(max_depth=3),
            **params, random_state=RANDOM_STATE
        )
        pipe = construir_pipeline(modelo, 'smote')
        return cross_validate(pipe, X_train_prep, y_train, cv=CV,
                              scoring='roc_auc', n_jobs=-1)['test_score'].mean()

    mejores_params = dict(PARAMS_OPTUNA)
    for nombre, objetivo in [('Bagging', objetivo_bagging), ('AdaBoost', objetivo_adaboost)]:
        print(f'🔍 Optuna {nombre} ({N_TRIALS_OPTUNA} trials)...', end=' ', flush=True)
        t0    = time.time()
        study = optuna.create_study(direction='maximize',
                                    sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE))
        study.optimize(objetivo, n_trials=N_TRIALS_OPTUNA, show_progress_bar=False)
        AUC_OPTUNA[nombre]       = round(study.best_value, 4)
        mejores_params[nombre]   = study.best_params
        print(f'AUC={study.best_value:.4f}  ({time.time()-t0:.0f}s)')
        print(f'   Params: {study.best_params}')

    PARAMS_OPTUNA = mejores_params
    print('\n📋 Copia estos params en PARAMS_OPTUNA y pon FORZAR_OPTUNA = False')


⚠️  Params iniciales — ejecutar Optuna la primera vez (FORZAR_OPTUNA = True)
   Bagging y AdaBoost serán optimizados. Voting y Stacking usan defaults.
   Voting      : defaults (sin Optuna)
   Stacking    : defaults (sin Optuna)
   Bagging     : {'n_estimators': 100, 'max_samples': 0.8, 'max_features': 1.0}
   AdaBoost    : {'n_estimators': 200, 'learning_rate': 0.1}


In [6]:
# ============================================================
# CELDA 4b: LIMPIEZA DE PKL OBSOLETOS
# ============================================================
import glob, os, json as _json, joblib as _jl

MODELOS_FAMILIA_LIMPIEZA = ['Voting', 'Stacking', 'Bagging', 'AdaBoost']

ruta_meta = RUTA_MODELADO / 'meta_preparacion.json'
n_features_actual = None
if ruta_meta.exists():
    with open(ruta_meta) as _f:
        n_features_actual = _json.load(_f).get('n_features_final')

eliminados = []
for pkl in glob.glob(str(RUTA_MODELS / '*.pkl')):
    nombre = os.path.basename(pkl)
    if any(m in nombre for m in MODELOS_FAMILIA_LIMPIEZA):
        try:
            modelo = _jl.load(pkl)
            ultimo = modelo.steps[-1][1] if hasattr(modelo,'steps') else modelo
            n_pkl = getattr(ultimo, 'n_features_in_', None)
            if n_pkl is not None and n_features_actual is not None and n_pkl != n_features_actual:
                os.remove(pkl); eliminados.append(f'{nombre} ({n_pkl}→obsoleto)')
        except Exception:
            os.remove(pkl); eliminados.append(f'{nombre} (no cargable)')

# Caso crítico: parquet existe pero no hay pkl → desincronizado → borrar parquet
_pkls_familia = [p for p in glob.glob(str(RUTA_MODELS / '*.pkl'))
                 if any(m in os.path.basename(p) for m in MODELOS_FAMILIA_LIMPIEZA)]
if not _pkls_familia:
    _ruta_pq = RUTA_RESULTS / 'results_ensambles.parquet'
    if _ruta_pq.exists():
        os.remove(_ruta_pq)
        print(f'⚠️  Sin PKL pero parquet existía → eliminado para re-entrenar completo')

if eliminados:
    print(f'⚠️  PKL eliminados: {eliminados}')
    _ruta_pq2 = RUTA_RESULTS / 'results_ensambles.parquet'
    if _ruta_pq2.exists():
        os.remove(_ruta_pq2)
        print(f'   Parquet eliminado → re-entrenamiento completo')
else:
    print(f'✅ PKL OK con {n_features_actual} features')


✅ PKL OK con 27 features


In [7]:
# ============================================================================
# CELDA 5: ENTRENAMIENTO — 12 COMBINACIONES (4 modelos × 3 estrategias)
# ============================================================================
# Disk-check: carga pkl si existen, reconstruye métricas si hay pkl sin parquet.
# ============================================================================

def construir_modelo(nombre: str, estrategia: str):
    """Instancia ensamble con hiperparámetros registrados."""
    p = mejores_params.get(nombre, {})

    if nombre == 'Voting':
        # Soft voting — promedio de probabilidades de los modelos base
        return VotingClassifier(
            estimators=modelos_base,
            voting='soft',
            n_jobs=-1
        )
    elif nombre == 'Stacking':
        # Meta-modelo LogReg que aprende a combinar predicciones base
        return StackingClassifier(
            estimators=modelos_base,
            final_estimator=LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
            cv=3,
            n_jobs=-1
        )
    elif nombre == 'Bagging':
        return BaggingClassifier(
            estimator=DecisionTreeClassifier(max_depth=5, random_state=RANDOM_STATE),
            n_estimators=p.get('n_estimators', 100),
            max_samples=p.get('max_samples', 0.8),
            max_features=p.get('max_features', 1.0),
            random_state=RANDOM_STATE,
            n_jobs=-1
        )
    elif nombre == 'AdaBoost':
        return AdaBoostClassifier(
            estimator=DecisionTreeClassifier(max_depth=3),
            n_estimators=p.get('n_estimators', 200),
            learning_rate=p.get('learning_rate', 0.1),
            random_state=RANDOM_STATE
        )


claves_esperadas = [f'{m}__{e}' for m in MODELOS_M06 for e in ESTRATEGIAS]
pkls_en_disco    = [c for c in claves_esperadas if (RUTA_MODELS / f'{c}.pkl').exists()]
parquet_en_disco = (RUTA_RESULTS / 'results_ensambles.parquet').exists()

print(f'📦 Modelos en disco : {len(pkls_en_disco)}/12')
print(f'📊 Parquet resultados: {parquet_en_disco}')

modelos_fit = {c: joblib.load(RUTA_MODELS / f'{c}.pkl') for c in pkls_en_disco}

if parquet_en_disco:
    df_res = pd.read_parquet(RUTA_RESULTS / 'results_ensambles.parquet')
    df_cv  = df_res.sort_values('auc_mean', ascending=False)
    emisiones = None
    print('✅ Resultados cargados desde disco')

elif len(pkls_en_disco) == 12:
    print('\n⏳ Reconstruyendo métricas (modelos en disco, parquet no)...')
    resultados_cv   = []
    resultados_test = []
    for nombre in MODELOS_M06:
        print(f'   {nombre}...', end=' ')
        for estrategia in ESTRATEGIAS:
            clave = f'{nombre}__{estrategia}'
            pipe  = modelos_fit[clave]
            res_cv = evaluar_cv(nombre, construir_modelo(nombre, estrategia),
                                X_train_prep, y_train, estrategia)
            resultados_cv.append(res_cv)
            res_test = calcular_metricas_test(nombre, pipe, X_test_prep, y_test, estrategia)
            resultados_test.append(res_test)
        print('✅')
    df_cv   = pd.DataFrame(resultados_cv).sort_values('auc_mean', ascending=False)
    df_test = pd.DataFrame(resultados_test)
    df_res  = df_cv.merge(df_test, on=['modelo', 'estrategia'])
    emisiones = None
    print('✅ Métricas reconstruidas')

else:
    print('\n⏳ Entrenando 12 combinaciones...')
    resultados_cv   = []
    resultados_test = []

    if CODECARBON_OK:
        tracker = EmissionsTracker(project_name=f'TFM_f5_m06_{FAMILIA}',
                                   output_dir=str(RUTA_RESULTS), log_level='error')
        tracker.start()

    print('=' * 60)
    print('ENTRENAMIENTO — 4 modelos × 3 estrategias = 12 combinaciones')
    print('=' * 60)
    for nombre in MODELOS_M06:
        print(f'  {nombre}...', end=' ', flush=True)
        for estrategia in ESTRATEGIAS:
            clave  = f'{nombre}__{estrategia}'
            modelo = construir_modelo(nombre, estrategia)
            res_cv = evaluar_cv(nombre, construir_modelo(nombre, estrategia),
                                X_train_prep, y_train, estrategia)
            resultados_cv.append(res_cv)
            pipe_final = construir_pipeline(modelo, estrategia)
            pipe_final.fit(X_train_prep, y_train)
            modelos_fit[clave] = pipe_final
            joblib.dump(pipe_final, RUTA_MODELS / f'{clave}.pkl')
            res_test = calcular_metricas_test(nombre, pipe_final,
                                              X_test_prep, y_test, estrategia)
            resultados_test.append(res_test)
        print('✅')

    if CODECARBON_OK:
        emisiones = tracker.stop()
        print(f'\n🌿 Emisiones CO₂: {emisiones:.6f} kg')
    else:
        emisiones = None

    df_cv   = pd.DataFrame(resultados_cv).sort_values('auc_mean', ascending=False)
    df_test = pd.DataFrame(resultados_test)
    df_res  = df_cv.merge(df_test, on=['modelo', 'estrategia'])
    print('\n✅ Entrenamiento completado')

MEJOR_MODELO     = df_cv.iloc[0]['modelo']
MEJOR_ESTRATEGIA = df_cv.iloc[0]['estrategia']
mejor_clave      = f'{MEJOR_MODELO}__{MEJOR_ESTRATEGIA}'
mejor_pipe       = modelos_fit[mejor_clave]

print(f'\n🏆 Mejor: {MEJOR_MODELO} + {MEJOR_ESTRATEGIA}')
print(f'   AUC CV = {df_cv.iloc[0]["auc_mean"]:.4f} ± {df_cv.iloc[0]["auc_std"]:.4f}')
print(f'   F1  CV = {df_cv.iloc[0]["f1_mean"]:.4f} ± {df_cv.iloc[0]["f1_std"]:.4f}')


📦 Modelos en disco : 2/12
📊 Parquet resultados: False


[codecarbon WARNING @ 21:23:53] Multiple instances of codecarbon are allowed to run at the same time.



⏳ Entrenando 12 combinaciones...
ENTRENAMIENTO — 4 modelos × 3 estrategias = 12 combinaciones
✅ Voting... 
✅ Stacking... 
✅ Bagging... 
✅ AdaBoost... 

🌿 Emisiones CO₂: 0.023487 kg

✅ Entrenamiento completado

🏆 Mejor: Stacking + none
   AUC CV = 0.9509 ± 0.0021
   F1  CV = 0.8252 ± 0.0043


In [8]:
# ============================================================================
# CELDA 6: TABLA DE RESULTADOS
# ============================================================================

cols_cv = ['modelo', 'estrategia', 'auc_mean', 'auc_std',
           'f1_mean', 'precision_mean', 'recall_mean', 'tiempo_s']

print('=' * 70)
print('RESULTADOS 5-Fold CV — ordenado por AUC')
print('=' * 70)
print(df_cv[cols_cv].to_string(index=False, float_format='{:.4f}'.format))

if 'auc_test' in df_res.columns:
    cols_test = ['modelo', 'estrategia', 'auc_test', 'auc_pr_test',
                 'f1_test', 'precision_test', 'recall_test', 'kappa_test']
    print('\n' + '=' * 70)
    print('MÉTRICAS EN TEST')
    print('=' * 70)
    print(df_res[cols_test].sort_values('auc_test', ascending=False)
          .to_string(index=False, float_format='{:.4f}'.format))


RESULTADOS 5-Fold CV — ordenado por AUC
  modelo estrategia  auc_mean  auc_std  f1_mean  precision_mean  recall_mean  tiempo_s
Stacking       none    0.9509   0.0021   0.8252          0.8561       0.7966 1390.1840
Stacking   balanced    0.9509   0.0021   0.8252          0.8561       0.7966 1442.5750
Stacking      smote    0.9489   0.0028   0.8232          0.8154       0.8313 6342.9060
  Voting       none    0.9450   0.0025   0.8158          0.8423       0.7913  656.5280
  Voting   balanced    0.9450   0.0025   0.8158          0.8423       0.7913  481.0120
  Voting      smote    0.9448   0.0026   0.8201          0.8148       0.8257 1731.1730
AdaBoost      smote    0.9299   0.0029   0.7858          0.7545       0.8201   71.4340
 Bagging       none    0.9282   0.0033   0.7680          0.8304       0.7151   20.7310
 Bagging   balanced    0.9282   0.0033   0.7680          0.8304       0.7151   18.0240
AdaBoost       none    0.9241   0.0033   0.7531          0.8313       0.6885   31.6110
Ada

In [9]:
# ============================================================================
# CELDA 7: ANÁLISIS MODELOS BASE — contribución en Voting y Stacking
# ============================================================================
# Exclusivo de ensambles: muestra qué modelos base contribuyen más al
# rendimiento del conjunto. Para Voting se compara AUC individual vs ensamble.
# Para Stacking se muestran los coeficientes del meta-modelo LogReg.
# ============================================================================

graficos_b64 = {}

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# --- Voting: AUC individual de cada modelo base vs AUC Voting ---
ax_v = axes[0]
nombres_base = [n for n, _ in modelos_base]
aucs_base    = []
print('Calculando AUC individual de modelos base...')
for nombre_b, pipe_b in modelos_base:
    try:
        cv_b = cross_validate(pipe_b, X_train_prep, y_train, cv=CV,
                              scoring='roc_auc', n_jobs=-1)
        aucs_base.append(cv_b['test_score'].mean())
        print(f'  {nombre_b:<12} AUC={cv_b["test_score"].mean():.4f}')
    except Exception as e:
        aucs_base.append(0.0)
        print(f'  {nombre_b:<12} ⚠️  {e}')

auc_voting = df_cv[df_cv['modelo'] == 'Voting']['auc_mean'].max()
nombres_plot = nombres_base + ['Voting']
aucs_plot    = aucs_base + [auc_voting]
colores_v    = [COLOR] * len(nombres_base) + ['#e53e3e']

bars_v = ax_v.bar(nombres_plot, aucs_plot, color=colores_v, edgecolor='white')
ymin = max(0.5, min(aucs_plot) - 0.02)
ax_v.set_ylim(ymin, min(1.0, max(aucs_plot) + 0.02))
ax_v.set_ylabel('AUC-ROC (5-Fold CV)')
ax_v.set_title('Modelos base vs VotingClassifier\nRojo = Voting ensamblado', fontweight='bold')
for bar, val in zip(bars_v, aucs_plot):
    ax_v.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001,
              f'{val:.4f}', ha='center', va='bottom', fontsize=9)
ax_v.spines['top'].set_visible(False)
ax_v.spines['right'].set_visible(False)

# --- Stacking: coeficientes del meta-modelo LogReg ---
ax_s = axes[1]
try:
    mejor_est_stack = (df_cv[df_cv['modelo'] == 'Stacking']
                       .sort_values('auc_mean', ascending=False).iloc[0]['estrategia'])
    pipe_stack  = modelos_fit[f'Stacking__{mejor_est_stack}']
    meta_modelo = pipe_stack.named_steps['model'].final_estimator_
    coefs       = meta_modelo.coef_[0]
    ax_s.barh(nombres_base, np.abs(coefs),
              color=[COLOR if c >= 0 else '#e53e3e' for c in coefs],
              height=0.5)
    ax_s.set_xlabel('|Coeficiente| meta-modelo LogReg')
    ax_s.set_title(f'Stacking — peso de cada modelo base\n({mejor_est_stack})',
                   fontweight='bold')
except Exception as e:
    ax_s.text(0.5, 0.5, f'No disponible:\n{e}', ha='center', va='center',
              transform=ax_s.transAxes, fontsize=10)
    ax_s.set_title('Stacking — pesos meta-modelo', fontweight='bold')

ax_s.spines['top'].set_visible(False)
ax_s.spines['right'].set_visible(False)

plt.tight_layout()
graficos_b64['modelos_base'] = figura_a_base64(fig)
plt.close(fig)
print('\n✅ Análisis modelos base generado')


Calculando AUC individual de modelos base...
  LogReg       AUC=0.9100
  RF           AUC=0.9473
  CatBoost     AUC=0.9492
  EBM          AUC=0.9374

✅ Análisis modelos base generado


In [10]:
# ============================================================================
# CELDA 8: CURVA DE APRENDIZAJE
# ============================================================================

sss = StratifiedShuffleSplit(n_splits=1, test_size=0.70, random_state=RANDOM_STATE)
idx_sub, _ = next(sss.split(X_train_prep, y_train))
X_sub = X_train_prep.iloc[idx_sub]
y_sub = y_train.iloc[idx_sub]

print(f'Submuestra: {fmt(len(X_sub))} filas '
      f'({len(X_sub)/len(X_train_prep)*100:.0f}% del train)  '
      f'abandono={y_sub.mean()*100:.1f}%')
print('Calculando curva de aprendizaje...', end=' ', flush=True)
t0 = time.time()

train_sizes, train_scores, cv_scores = learning_curve(
    mejor_pipe, X_sub, y_sub,
    cv=CV, scoring='roc_auc',
    train_sizes=np.linspace(0.1, 1.0, 8),
    n_jobs=-1
)

train_mean = train_scores.mean(axis=1)
train_std  = train_scores.std(axis=1)
cv_mean    = cv_scores.mean(axis=1)
cv_std     = cv_scores.std(axis=1)

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(train_sizes, train_mean, 'o-', color=COLOR, label='Train AUC', linewidth=2)
ax.fill_between(train_sizes, train_mean - train_std, train_mean + train_std,
                alpha=0.15, color=COLOR)
ax.plot(train_sizes, cv_mean, 'o-', color='#e53e3e', label='CV AUC (5-fold)', linewidth=2)
ax.fill_between(train_sizes, cv_mean - cv_std, cv_mean + cv_std,
                alpha=0.15, color='#e53e3e')
ax.set_xlabel('Tamaño del conjunto de entrenamiento (submuestra 30%)')
ax.set_ylabel('AUC-ROC')
ax.set_title(f'Curva de aprendizaje — {MEJOR_MODELO} + {MEJOR_ESTRATEGIA}\n'
             f'Submuestra estratificada 30% del train',
             fontsize=11, fontweight='bold')
ax.legend()
ax.set_ylim(0.5, 1.0)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
graficos_b64['curva_aprendizaje'] = figura_a_base64(fig)
plt.close(fig)

gap = train_mean[-1] - cv_mean[-1]
print(f'✅  ({time.time()-t0:.0f}s)')
print(f'Gap train-CV: {gap:.4f}  → '
      f'{"Posible overfitting" if gap > 0.05 else "Generalización correcta"}')


Submuestra: 8.068 filas (30% del train)  abandono=29.3%
✅  (3239s) curva de aprendizaje... 
Gap train-CV: 0.0370  → Generalización correcta


In [11]:
# ============================================================================
# CELDA 9: CALIBRACIÓN + ROC COMPARATIVA
# ============================================================================

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
colores_4 = ['#3182ce', '#e53e3e', '#38a169', '#d69e2e']

ax_cal = axes[0]
ax_cal.plot([0,1],[0,1],'k--',linewidth=1.2,label='Calibración perfecta')
ax_roc = axes[1]
ax_roc.plot([0,1],[0,1],'k--',linewidth=1)

for idx, nombre_m in enumerate(MODELOS_M06):
    mejor_est = (df_cv[df_cv['modelo'] == nombre_m]
                 .sort_values('auc_mean', ascending=False).iloc[0]['estrategia'])
    pipe_m = modelos_fit[f'{nombre_m}__{mejor_est}']
    try:
        y_proba = pipe_m.predict_proba(X_test_prep)[:, 1]
    except Exception:
        continue

    color_m = colores_4[idx % len(colores_4)]
    frac_pos, mean_pred = calibration_curve(y_test, y_proba, n_bins=10)
    ax_cal.plot(mean_pred, frac_pos, 'o-', color=color_m,
                label=f'{nombre_m} ({mejor_est})', linewidth=1.8, markersize=5)

    fpr, tpr, _ = roc_curve(y_test, y_proba)
    auc_v = roc_auc_score(y_test, y_proba)
    ax_roc.plot(fpr, tpr, color=color_m,
                label=f'{nombre_m} AUC={auc_v:.3f}', linewidth=2)

for ax, titulo in zip([ax_cal, ax_roc], ['Calibración de probabilidades', 'Curvas ROC']):
    ax.set_title(titulo, fontweight='bold')
    ax.legend(fontsize=9)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

ax_cal.set_xlabel('Prob. predicha'); ax_cal.set_ylabel('Fracción positivos reales')
ax_roc.set_xlabel('FPR'); ax_roc.set_ylabel('TPR')

plt.tight_layout()
graficos_b64['calibracion_roc'] = figura_a_base64(fig)
plt.close(fig)
print('✅ Calibración + ROC generados')


✅ Calibración + ROC generados


In [12]:
# ============================================================================
# CELDA 10: PR CURVE + F1 POR UMBRAL
# ============================================================================

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
ax_pr, ax_f1 = axes
umbrales_rango = np.arange(0.05, 0.96, 0.01)

baseline_pr = y_test.mean()
ax_pr.axhline(baseline_pr, color='gray', linestyle='--', linewidth=1,
              label=f'Baseline ({baseline_pr:.2f})')

for idx, nombre_m in enumerate(MODELOS_M06):
    mejor_est = (df_cv[df_cv['modelo'] == nombre_m]
                 .sort_values('auc_mean', ascending=False).iloc[0]['estrategia'])
    pipe_m = modelos_fit[f'{nombre_m}__{mejor_est}']
    try:
        y_proba = pipe_m.predict_proba(X_test_prep)[:, 1]
    except Exception:
        continue

    color_m = colores_4[idx % len(colores_4)]
    prec, rec, _ = precision_recall_curve(y_test, y_proba)
    auc_pr = average_precision_score(y_test, y_proba)
    ax_pr.plot(rec, prec, color=color_m, linewidth=2,
               label=f'{nombre_m} AUC-PR={auc_pr:.3f}')

    f1s = [f1_score(y_test, (y_proba >= t).astype(int), zero_division=0)
           for t in umbrales_rango]
    ax_f1.plot(umbrales_rango, f1s, color=color_m, linewidth=2,
               label=f'{nombre_m} F1_max={max(f1s):.3f}')

for ax, titulo in zip([ax_pr, ax_f1],
                      ['Curva PR (desbalance 70/30)', 'F1 por umbral']):
    ax.set_title(titulo, fontweight='bold')
    ax.legend(fontsize=8)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

ax_pr.set_xlabel('Recall'); ax_pr.set_ylabel('Precision')
ax_f1.set_xlabel('Umbral'); ax_f1.set_ylabel('F1-score')
ax_f1.axvline(0.5, color='gray', linestyle='--', linewidth=1)

plt.tight_layout()
graficos_b64['pr_f1_umbral'] = figura_a_base64(fig)
plt.close(fig)
print('✅ PR curve + F1 por umbral generados')


✅ PR curve + F1 por umbral generados


In [13]:
# ============================================================================
# CELDA 11: MATRIZ DE CONFUSIÓN + COMPARATIVA AUC
# ============================================================================

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

y_pred_mejor = mejor_pipe.predict(X_test_prep)
cm = confusion_matrix(y_test, y_pred_mejor)
ax_cm = axes[0]
im = ax_cm.imshow(cm, interpolation='nearest', cmap='Blues')
plt.colorbar(im, ax=ax_cm)
clases = ['Continúa (0)', 'Abandona (1)']
ax_cm.set_xticks([0,1]); ax_cm.set_xticklabels(clases, rotation=30, ha='right')
ax_cm.set_yticks([0,1]); ax_cm.set_yticklabels(clases)
thresh = cm.max() / 2
for i in range(2):
    for j in range(2):
        ax_cm.text(j, i, f'{cm[i,j]:,}', ha='center', va='center',
                   color='white' if cm[i,j] > thresh else 'black',
                   fontsize=13, fontweight='bold')
ax_cm.set_title(f'Matriz de confusión\n{MEJOR_MODELO} + {MEJOR_ESTRATEGIA}',
                fontweight='bold')
ax_cm.set_ylabel('Real'); ax_cm.set_xlabel('Predicho')

ax_bar = axes[1]
aucs_cv_bar = [df_cv[df_cv['modelo']==m]['auc_mean'].max() for m in MODELOS_M06]
bars = ax_bar.bar(MODELOS_M06, aucs_cv_bar, color=colores_4, edgecolor='white')
ymin = max(0.5, min(aucs_cv_bar) - 0.02)
ax_bar.set_ylim(ymin, min(1.0, max(aucs_cv_bar) + 0.02))
ax_bar.set_ylabel('AUC-ROC (mejor estrategia, CV)')
ax_bar.set_title('Comparativa AUC — 4 ensambles', fontweight='bold')
for bar, val in zip(bars, aucs_cv_bar):
    ax_bar.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.001,
                f'{val:.4f}', ha='center', va='bottom', fontsize=10)
ax_bar.spines['top'].set_visible(False)
ax_bar.spines['right'].set_visible(False)

plt.tight_layout()
graficos_b64['cm_comparativa'] = figura_a_base64(fig)
plt.close(fig)

print('✅ Confusión + comparativa AUC generados')
print()
print(classification_report(y_test, y_pred_mejor, target_names=['Continúa','Abandona']))


✅ Confusión + comparativa AUC generados

              precision    recall  f1-score   support

    Continúa       0.92      0.94      0.93      4758
    Abandona       0.85      0.81      0.83      1967

    accuracy                           0.90      6725
   macro avg       0.89      0.87      0.88      6725
weighted avg       0.90      0.90      0.90      6725



In [14]:
# ============================================================================
# CELDA 12: AJUSTE DE UMBRAL ÓPTIMO
# ============================================================================

RECALL_MINIMO   = 0.75
umbrales_result = []
umbrales_rango  = np.arange(0.10, 0.91, 0.01)

print('=' * 60)
print('AJUSTE DE UMBRAL — búsqueda sobre test set')
print('=' * 60)

for nombre in MODELOS_M06:
    mejor_est = (df_cv[df_cv['modelo'] == nombre]
                 .sort_values('auc_mean', ascending=False).iloc[0]['estrategia'])
    pipe_m  = modelos_fit[f'{nombre}__{mejor_est}']
    y_proba = pipe_m.predict_proba(X_test_prep)[:, 1]

    f1s    = [f1_score(y_test, (y_proba >= t).astype(int), zero_division=0)
              for t in umbrales_rango]
    idx_f1 = int(np.argmax(f1s))
    u_f1   = round(float(umbrales_rango[idx_f1]), 2)
    f1_opt = round(float(f1s[idx_f1]), 4)

    recalls  = [recall_score(y_test, (y_proba >= t).astype(int), zero_division=0)
                for t in umbrales_rango]
    idx_rec  = next((j for j, r in enumerate(recalls) if r >= RECALL_MINIMO), None)
    u_rec    = round(float(umbrales_rango[idx_rec]), 2) if idx_rec is not None else None
    rec_rec  = round(float(recalls[idx_rec]), 4)        if idx_rec is not None else None
    pre_rec  = round(float(precision_score(
                   y_test, (y_proba >= umbrales_rango[idx_rec]).astype(int),
                   zero_division=0)), 4)                if idx_rec is not None else None

    y05    = (y_proba >= 0.50).astype(int)
    f1_05  = round(float(f1_score(y_test, y05, zero_division=0)), 4)
    rec_05 = round(float(recall_score(y_test, y05, zero_division=0)), 4)
    ganancia = round(f1_opt - f1_05, 4)

    umbrales_result.append(dict(
        modelo=nombre, estrategia=mejor_est,
        umbral_f1=u_f1, f1_umbral_opt=f1_opt, f1_umbral_05=f1_05,
        ganancia_f1=ganancia, umbral_recall=u_rec,
        recall_garantizado=rec_rec, precision_recall=pre_rec,
        recall_umbral_05=rec_05,
    ))

    print(f'  {nombre} ({mejor_est})')
    print(f'    Umbral 0.50     → F1={f1_05}  Recall={rec_05}')
    print(f'    Umbral ópt. F1 → {u_f1}  F1={f1_opt}  (+{ganancia})')
    if u_rec:
        print(f'    Umbral Recall≥{RECALL_MINIMO} → {u_rec}  Recall={rec_rec}  Prec={pre_rec}')
    print()

df_umbrales = pd.DataFrame(umbrales_result)
print('✅ Análisis de umbral completado')


AJUSTE DE UMBRAL — búsqueda sobre test set
  Voting (none)
    Umbral 0.50     → F1=0.8176  Recall=0.8043
    Umbral ópt. F1 → 0.45  F1=0.8245  (+0.0069)
    Umbral Recall≥0.75 → 0.1  Recall=0.9705  Prec=0.5123

  Stacking (none)
    Umbral 0.50     → F1=0.8273  Recall=0.8073
    Umbral ópt. F1 → 0.49  F1=0.8294  (+0.0021)
    Umbral Recall≥0.75 → 0.1  Recall=0.9309  Prec=0.6651

  Bagging (none)
    Umbral 0.50     → F1=0.7713  Recall=0.7331
    Umbral ópt. F1 → 0.34  F1=0.7995  (+0.0282)
    Umbral Recall≥0.75 → 0.1  Recall=0.9527  Prec=0.4984

  AdaBoost (smote)
    Umbral 0.50     → F1=0.7861  Recall=0.8363
    Umbral ópt. F1 → 0.53  F1=0.7903  (+0.0042)
    Umbral Recall≥0.75 → 0.1  Recall=1.0  Prec=0.2925

✅ Análisis de umbral completado


In [15]:
# ============================================================================
# CELDA 13: SERIALIZACIÓN DE RESULTADOS
# ============================================================================

if not (RUTA_RESULTS / 'results_ensambles.parquet').exists():
    df_res.to_parquet(RUTA_RESULTS / 'results_ensambles.parquet', index=False)
    print('✅ results_ensambles.parquet guardado')
else:
    print('✅ results_ensambles.parquet ya existía')

df_res.to_excel(RUTA_RESULTS / 'results_ensambles.xlsx', index=False)
print('✅ results_ensambles.xlsx guardado')

meta = {
    'fecha'            : datetime.now().isoformat(),
    'familia'          : FAMILIA,
    'modelos'          : MODELOS_M06,
    'estrategias'      : ESTRATEGIAS,
    'n_combinaciones'  : 12,
    'n_trials_optuna'  : N_TRIALS_OPTUNA,
    'cv_folds'         : N_SPLITS_CV,
    'random_state'     : RANDOM_STATE,
    'mejor_modelo'     : MEJOR_MODELO,
    'mejor_estrategia' : MEJOR_ESTRATEGIA,
    'mejor_auc_cv'     : round(float(df_cv.iloc[0]['auc_mean']), 4),
    'mejor_f1_cv'      : round(float(df_cv.iloc[0]['f1_mean']), 4),
    'mejores_params'   : PARAMS_OPTUNA,
    'auc_optuna'       : AUC_OPTUNA,
    'modelos_base'     : [n for n, _ in modelos_base],
    'emisiones_co2_kg' : emisiones,
}
with open(RUTA_RESULTS / 'results_ensambles.json', 'w', encoding='utf-8') as f:
    json.dump(meta, f, indent=2, ensure_ascii=False, default=str)

print('✅ results_ensambles.json guardado')
print(f'✅ {len(modelos_fit)} modelos .pkl en {RUTA_MODELS}')


✅ results_ensambles.parquet guardado
✅ results_ensambles.xlsx guardado
✅ results_ensambles.json guardado
✅ 12 modelos .pkl en C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\05_modelado\models


In [16]:
# ============================================================================
# CELDA 14: GENERAR HTML
# ============================================================================

RUTA_HTML_SALIDA = RUTA_HTML_F5 / 'm06_ensambles.html'

# KPIs
kpis = [
    {'valor': '4',                                  'titulo': 'Modelos'},
    {'valor': '3',                                  'titulo': 'Estrategias'},
    {'valor': '12',                                 'titulo': 'Combinaciones'},
    {'valor': '20 (Bag+Ada)',                        'titulo': 'Trials Optuna'},
    {'valor': f'{df_cv.iloc[0]["auc_mean"]:.3f}',   'titulo': 'Mejor AUC CV'},
    {'valor': f'{df_cv.iloc[0]["f1_mean"]:.3f}',    'titulo': 'Mejor F1 CV'},
]
kpis_html = '<div style="display:flex;flex-wrap:wrap;gap:16px;margin:16px 0;">' + ''.join(
    f'<div style="background:#f7fafc;border:1px solid #e2e8f0;border-radius:10px;'
    f'padding:18px 28px;text-align:center;min-width:120px;">'
    f'<div style="font-size:2rem;font-weight:700;color:#3182ce;">{k["valor"]}</div>'
    f'<div style="font-size:0.85rem;color:#555;margin-top:4px;">{k["titulo"]}</div>'
    f'</div>'
    for k in kpis
) + '</div>'

# Texto dinámico
segundo  = df_cv.iloc[1]
diff_auc = df_cv.iloc[0]['auc_mean'] - segundo['auc_mean']
diff_f1  = df_cv.iloc[0]['f1_mean']  - segundo['f1_mean']

texto_dinamico = (
    f'<p>El análisis de <strong>12 combinaciones</strong> '
    f'(4 ensambles × 3 estrategias) revela que '
    f'<strong>{MEJOR_MODELO}</strong> con estrategia <strong>{MEJOR_ESTRATEGIA}</strong> '
    f'alcanza el mejor AUC CV: '
    f'<strong>{df_cv.iloc[0]["auc_mean"]:.4f} ± {df_cv.iloc[0]["auc_std"]:.4f}</strong> '
    f'(F1={df_cv.iloc[0]["f1_mean"]:.4f}).</p>'
    f'<p>Supera al segundo clasificador ({segundo["modelo"]} + {segundo["estrategia"]}) en '
    f'<strong>{diff_auc:.4f} puntos de AUC</strong> y <strong>{diff_f1:.4f} de F1</strong>.</p>'
    f'<p><strong>Sobre los ensambles:</strong> Voting y Stacking dependen de la diversidad '
    f'y calidad de los modelos base (LogReg, RF, CatBoost, EBM de m01–m05). '
    f'Bagging reduce varianza entrenando subconjuntos con base DecisionTree. '
    f'AdaBoost corrige errores secuencialmente ponderando los ejemplos difíciles.</p>'
)

# Tabla pivotada
filas_pivot = ''
for modelo in MODELOS_M06:
    sub      = df_cv[df_cv['modelo'] == modelo].sort_values('auc_mean', ascending=False).iloc[0]
    es_mejor = modelo == MEJOR_MODELO
    bg       = 'background:#ebf8ff;font-weight:600;' if es_mejor else ''
    estrella = ' 🏆' if es_mejor else ''
    auc_opt  = AUC_OPTUNA.get(modelo)
    auc_opt_str = f'{auc_opt:.4f}' if auc_opt else '— (sin Optuna)'
    params_str = json.dumps(PARAMS_OPTUNA.get(modelo, {})) if PARAMS_OPTUNA.get(modelo) else 'defaults'
    filas_pivot += (
        f'<tr style="{bg}"><td>{modelo}{estrella}</td>'
        f'<td>{sub["estrategia"]}</td>'
        f'<td>{sub["auc_mean"]:.4f} ± {sub["auc_std"]:.4f}</td>'
        f'<td>{sub["f1_mean"]:.4f}</td>'
        f'<td>{sub["precision_mean"]:.4f}</td>'
        f'<td>{sub["recall_mean"]:.4f}</td>'
        f'<td>{auc_opt_str}</td>'
        f'<td><code>{params_str}</code></td></tr>'
    )
tabla_pivot = (
    '<p>Una fila por modelo con su mejor estrategia. 🏆 = mejor combinación global.</p>'
    '<table class="tabla-datos"><thead><tr>'
    '<th>Modelo</th><th>Mejor estrategia</th>'
    '<th>AUC CV (mean±std)</th><th>F1 CV</th>'
    '<th>Precisión</th><th>Recall</th>'
    '<th>AUC Optuna</th><th>Hiperparámetros</th>'
    f'</tr></thead><tbody>{filas_pivot}</tbody></table>'
)

# Tabla CV completa
filas_cv = ''
for _, r in df_cv.iterrows():
    es_mejor = r['modelo'] == MEJOR_MODELO and r['estrategia'] == MEJOR_ESTRATEGIA
    bg = 'background:#ebf8ff;font-weight:600;' if es_mejor else ''
    filas_cv += (
        f'<tr style="{bg}"><td>{r["modelo"]}</td><td>{r["estrategia"]}</td>'
        f'<td>{r["auc_mean"]:.4f} ± {r["auc_std"]:.4f}</td>'
        f'<td>{r["f1_mean"]:.4f}</td><td>{r["precision_mean"]:.4f}</td>'
        f'<td>{r["recall_mean"]:.4f}</td><td>{r["tiempo_s"]:.1f}s</td></tr>'
    )
tabla_cv = (
    f'<p>Ordenado por AUC-ROC. Destacado = mejor: '
    f'<strong>{MEJOR_MODELO} + {MEJOR_ESTRATEGIA}</strong>.</p>'
    '<table class="tabla-datos"><thead><tr>'
    '<th>Modelo</th><th>Estrategia</th><th>AUC (mean±std)</th>'
    '<th>F1</th><th>Precisión</th><th>Recall</th><th>Tiempo</th>'
    f'</tr></thead><tbody>{filas_cv}</tbody></table>'
)

co2_html = (
    f'<p>🌿 Huella CO₂: <strong>{emisiones:.6f} kg CO₂eq</strong></p>'
    if emisiones
    else '<p>ℹ️ Modelos cargados desde disco — sin nuevo entrenamiento.</p>'
)

def img_tag(key, caption=''):
    b64 = graficos_b64.get(key)
    if not b64:
        return '<p><em>Gráfico no disponible</em></p>'
    cap = f'<p style="font-size:0.85rem;color:#666;margin-top:4px;">{caption}</p>' if caption else ''
    return f'<img src="data:image/png;base64,{b64}" style="max-width:100%;border-radius:8px;">{cap}'

secciones = (
    '<section class="seccion"><h2>Resumen</h2>' + kpis_html + '</section>'
    '<section class="seccion"><h2>Conclusiones del análisis</h2>' + texto_dinamico + '</section>'
    '<section class="seccion"><h2>Comparativa por modelo — mejor estrategia</h2>' + tabla_pivot + '</section>'
    '<section class="seccion"><h2>Resultados completos 5-Fold CV — 12 combinaciones</h2>' + tabla_cv + '</section>'
    '<section class="seccion"><h2>Modelos base — contribución en Voting y Stacking</h2>'
    '<p>AUC individual de cada modelo base vs AUC del Voting ensamblado (izquierda). '
    'Peso de cada modelo en el meta-modelo LogReg del Stacking (derecha).</p>'
    + img_tag('modelos_base') + '</section>'
    f'<section class="seccion"><h2>Curva de aprendizaje — {MEJOR_MODELO}</h2>'
    + img_tag('curva_aprendizaje',
              'Submuestra 30% del train. Convergencia train/CV indica buena generalización.') + '</section>'
    '<section class="seccion"><h2>Calibración de probabilidades y curvas ROC</h2>'
    + img_tag('calibracion_roc') + '</section>'
    '<section class="seccion"><h2>Curva PR + F1 por umbral</h2>'
    '<p>Con desbalance 70/30, AUC-PR es más informativo que AUC-ROC.</p>'
    + img_tag('pr_f1_umbral') + '</section>'
    '<section class="seccion"><h2>Matriz de confusión y comparativa AUC</h2>'
    + img_tag('cm_comparativa') + '</section>'
    '<section class="seccion"><h2>Sostenibilidad computacional</h2>' + co2_html + '</section>'
)

render_pagina_desde_fichero(
    'f5_m06_ensambles.ipynb',
    secciones
)
print(f'\n✅ HTML generado: {RUTA_HTML_SALIDA}')



✅ HTML generado: C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\docs\html\fase5\m06_ensambles.html


In [17]:
# ============================================================================
# CELDA 15: RESUMEN FINAL
# ============================================================================

print('=' * 60)
print('RESUMEN FINAL — f5_m06_ensambles')
print('=' * 60)
print(f'Familia     : Ensambles heterogéneos')
print(f'Modelos     : Voting · Stacking · Bagging · AdaBoost')
print(f'Estrategias : none · balanced · smote  (12 combinaciones)')
print(f'Optuna      : {N_TRIALS_OPTUNA} trials (Bagging + AdaBoost) | Voting + Stacking sin Optuna')
print(f'CV folds    : {N_SPLITS_CV}')
print(f'Modelos base: {[n for n, _ in modelos_base]}')
print()
print(f'🏆 Mejor: {MEJOR_MODELO} + {MEJOR_ESTRATEGIA}')
print(f'   AUC CV = {df_cv.iloc[0]["auc_mean"]:.4f} ± {df_cv.iloc[0]["auc_std"]:.4f}')
print(f'   F1  CV = {df_cv.iloc[0]["f1_mean"]:.4f} ± {df_cv.iloc[0]["f1_std"]:.4f}')
print()
print('📁 Archivos:')
print('   data/05_modelado/results/results_ensambles.parquet')
print('   data/05_modelado/results/results_ensambles.xlsx')
print('   data/05_modelado/results/results_ensambles.json')
print('   data/05_modelado/models/  (12 × .pkl)')
print('   docs/html/fase5/m06_ensambles.html')
print()
print('➡️  Siguiente: f5_m07_comparacion.ipynb')


RESUMEN FINAL — f5_m06_ensambles
Familia     : Ensambles heterogéneos
Modelos     : Voting · Stacking · Bagging · AdaBoost
Estrategias : none · balanced · smote  (12 combinaciones)
Optuna      : 20 trials (Bagging + AdaBoost) | Voting + Stacking sin Optuna
CV folds    : 5
Modelos base: ['LogReg', 'RF', 'CatBoost', 'EBM']

🏆 Mejor: Stacking + none
   AUC CV = 0.9509 ± 0.0021
   F1  CV = 0.8252 ± 0.0043

📁 Archivos:
   data/05_modelado/results/results_ensambles.parquet
   data/05_modelado/results/results_ensambles.xlsx
   data/05_modelado/results/results_ensambles.json
   data/05_modelado/models/  (12 × .pkl)
   docs/html/fase5/m06_ensambles.html

➡️  Siguiente: f5_m07_comparacion.ipynb
